### Attribute Lookup 🔍

In [2]:
# When you write:

# obj.name

# Python has to figure out:

# "Where is name?"

# It doesn't simply look inside obj.

In [3]:
# For example:

class A:
    x = 10

a = A()

print(a.x)

# a doesn't have x in its own __dict__.

# Yet:
# a.x

# returns:
# 10

# because Python searches the class after checking the instance.

10


In [4]:
# a.x
#  ↓
# a.__dict__?
#  ↓
# not found
#  ↓
# A.__dict__?
#  ↓
# found x = 10
#  ↓
# 10

In [5]:
# But the real lookup rules are more sophisticated, especially when descriptors and inheritance are involved.

### __getattribute__, __getattr__, and Descriptors

In [ ]:
# __getattribute__ vs __getattr__ 🔍

In [ ]:
# __getattribute__

# __getattribute__ is called for essentially every normal attribute access.

In [7]:
class Person:

    def __init__(self):
        self.name = "Shri"

    def __getattribute__(self, name):
        print("Accessing:", name)
        return super().__getattribute__(name)

In [8]:
# Now:

p = Person()

print(p.name)

Accessing: name
Shri


In [ ]:
# Output:

# Accessing: name
# Shri

# Because:

# p.name

# goes through:

# p.__getattribute__("name")

In [ ]:
# ⚠️ Why use super() here?

# Inside __getattribute__, if you do:

# return self.name

# you create a problem:

# self.name
#  ↓
# __getattribute__
#  ↓
# self.name
#  ↓
# __getattribute__
#  ↓
# self.name
#  ↓
# ...

# That's infinite recursion.

# So we normally use:
# super().__getattribute__(name)

# to perform the normal attribute lookup.

In [10]:
# __getattr__

# __getattr__ is different.
# It is called only when normal attribute lookup fails.
class Person:

    def __init__(self):
        self.name = "Shri"

    def __getattr__(self, name):
        return "Attribute not found"

In [11]:
p = Person()

print(p.name)
print(p.age)

Shri
Attribute not found


In [ ]:
# Output:

# Shri
# Attribute not found

# Why?

# p.name
# p.name
#  ↓
# normal lookup succeeds
#  ↓
# "Shri"

# __getattr__ isn't called.

# p.age
# p.age
#  ↓
# normal lookup fails
#  ↓
# __getattr__("age")
#  ↓
# "Attribute not found"

In [ ]:
# 🧠 The difference
# __getattribute__
# → called for attribute access

# __getattr__
# → called only when normal lookup fails

# Think of it like:

# obj.name
#    ↓
# __getattribute__("name")
#    ↓
# Found?
#  ┌───────┴───────┐
#  YES             NO
#  ↓                ↓
# return it     __getattr__("name")

# This distinction is very important.

### Descriptors — the engine behind property

In [12]:
# You've already used:

# @property
# def age(self):
#     ...

# A descriptor is the mechanism that allows Python to customize attribute access.

In [13]:
# A descriptor is an object that defines methods such as:

# __get__
# __set__
# __delete__

In [14]:
class Descriptor:

    def __get__(self, instance, owner):
        print("Getting value")
        return 100


In [15]:
# Then:

class Person:
    age = Descriptor()


p = Person()

print(p.age)

Getting value
100


In [ ]:
# Python sees:

# p.age

# and the descriptor can intercept that access:

# p.age
#  ↓
# Descriptor.__get__()
#  ↓
# 100

# So descriptors are one of the mechanisms behind powerful Python features such as:

# @property
# classmethod
# staticmethod

# and many frameworks use them heavily.

### Descriptors — __get__ 🔥

In [ ]:
# A descriptor is an object that controls how an attribute is accessed.

In [16]:
class Descriptor:

    def __get__(self, instance, owner):
        print("Getting value")
        return 100


class Person:
    age = Descriptor()


p = Person()

print(p.age)

Getting value
100


In [ ]:
# Output:

# Getting value
# 100

# What's happening?
# Normally you think:

# p.age

# means:
# Look for age inside p.

# But age isn't actually stored in p.__dict__.

In [ ]:
# Instead:

# Person
# └── age → Descriptor object

# When Python sees:

# p.age

# it recognizes that age is a descriptor and calls:

# Descriptor.__get__(p, Person)

# So:

# p.age
#  ↓
# Descriptor.__get__()
#  ↓
# 100